# 02 - Bronze Layer: Raw Data Ingestion

## Project
Ubuntu Retail Group - Legacy System to Modern Analytics Platform Migration

## Objective
Ingest the four original System A CSV exports into the Bronze layer.

## Requirements
- Preserve source data without business transformations.
- Add ingested_at, source_file and batch_id metadata columns.
- Store the four datasets as Delta tables in Unity Catalog.
- Record and verify ingestion row counts.

## Target Tables
- workspace.default.bronze_customers
- workspace.default.bronze_products
- workspace.default.bronze_orders
- workspace.default.bronze_transactions

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

In [0]:
# Source landing zone
landing_path = "/Volumes/workspace/default/urg_data_landing/"

# Source-to-target configuration
source_files = {
    "customers": "customers_legacy_large.csv",
    "products": "products_legacy_large.csv",
    "orders": "orders_legacy_large.csv",
    "transactions": "transactions_legacy_large.csv"
}

# Target catalog and schema
target_catalog = "workspace"
target_schema = "default"

# Generate one unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

print("Ingestion batch ID:", batch_id)

In [0]:
raw_datasets = {}

for dataset_name, file_name in source_files.items():

    file_path = landing_path + file_name

    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .load(file_path)
    )

    raw_datasets[dataset_name] = df

    print(f"{dataset_name} loaded successfully")
    print(f"Source file: {file_name}")
    print(f"Rows: {df.count():,}")

In [0]:
def add_bronze_metadata(df, source_file, batch_id):

    bronze_df = (
        df
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("source_file", F.lit(source_file))
        .withColumn("batch_id", F.lit(batch_id))
    )

    return bronze_df

In [0]:
bronze_tables = {}

for dataset_name, df in raw_datasets.items():

    source_file = source_files[dataset_name]

    # Add ingestion metadata
    bronze_df = add_bronze_metadata(
        df,
        source_file,
        batch_id
    )

    # Define target table name
    table_name = f"{target_catalog}.{target_schema}.bronze_{dataset_name}"

    # Write as a managed Delta table
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    bronze_tables[dataset_name] = table_name

    print(f"Successfully written: {table_name}")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

In [0]:
for dataset_name, table_name in bronze_tables.items():

    print(f"\n{dataset_name.upper()} BRONZE SCHEMA")

    spark.table(table_name).printSchema()

In [0]:
count_results = []

for dataset_name, source_df in raw_datasets.items():

    table_name = bronze_tables[dataset_name]

    source_count = source_df.count()
    bronze_count = spark.table(table_name).count()

    count_results.append((
        dataset_name,
        source_count,
        bronze_count,
        source_count == bronze_count
    ))

count_df = spark.createDataFrame(
    count_results,
    [
        "dataset",
        "source_count",
        "bronze_count",
        "counts_match"
    ]
)

display(count_df)

In [0]:
for dataset_name, table_name in bronze_tables.items():

    print(f"\n{dataset_name.upper()} METADATA CHECK")

    display(
        spark.table(table_name)
        .select(
            "ingested_at",
            "source_file",
            "batch_id"
        )
        .distinct()
    )

In [0]:
display(spark.table("workspace.default.bronze_customers").limit(10))
display(spark.table("workspace.default.bronze_products").limit(10))
display(spark.table("workspace.default.bronze_orders").limit(10))
display(spark.table("workspace.default.bronze_transactions").limit(10))

## Bronze Layer Engineering Decisions

1. All four original CSV exports were loaded directly from the landing zone.
2. Schema inference was disabled to preserve source values as strings.
3. No business transformations, deduplication or cleaning operations were performed.
4. Three metadata columns were added:
   - ingested_at
   - source_file
   - batch_id
5. The datasets were stored as managed Delta tables in Unity Catalog.
6. Source and Bronze row counts were compared to verify ingestion completeness.

The Bronze layer acts as the raw historical reference for all subsequent
transformations in the Silver and Gold layers.